In [32]:
from Implementation_1 import smith_waterman as smith_waterman_sm
from Implementation_2 import smith_waterman as smith_waterman_tth
from Implementation_3 import align_sequences, traceback, fill_matrix_fast, decode_sequence, encode_seq


from time import time
from pprint import pprint

## Needleman-Wunsch

At first, we accidentally coded Needleman-Wunsch for global alignment instead of Smith-Waterman for local alignment. The difference between these is subtle.

- initialization: NW initializes a negative score matrix, whereas SW has a min of 0. Scores are calculated the same.
- traceback: NW starts in a fixed position (lower right), whereas SW starts at the argmax. NW terminates in the upper left whereas SW ends when the next alignment has a score of 0.

You can see the differences in the initalization below:

In [33]:
a = "GATNTANNNCA"
b = "TNACAGATTACA"

score_matrix_1 = fill_matrix_fast(a, b, nw=1)
print("=== NW score matrix ===")
print(score_matrix_1)

score_matrix_2 = fill_matrix_fast(a, b, nw=0)
print("=== SW score matrix ===")
print(score_matrix_2)


=== NW score matrix ===
[[  0  -1  -2  -3  -4  -5  -6  -7  -8  -9 -10 -11 -12]
 [ -1  -1  -2  -3  -4  -5  -4  -5  -6  -7  -8  -9 -10]
 [ -2  -2  -2  -1  -2  -3  -4  -3  -4  -5  -6  -7  -8]
 [ -3  -1  -2  -2  -2  -3  -4  -4  -2  -3  -4  -5  -6]
 [ -4  -2   0  -1  -2  -3  -4  -5  -3  -3  -4  -5  -6]
 [ -5  -3  -1  -1  -2  -3  -4  -5  -4  -2  -3  -4  -5]
 [ -6  -4  -2   0  -1  -1  -2  -3  -4  -3  -1  -2  -3]
 [ -7  -5  -3  -1  -1  -2  -2  -3  -4  -4  -2  -2  -3]
 [ -8  -6  -4  -2  -2  -2  -3  -3  -4  -5  -3  -3  -3]
 [ -9  -7  -5  -3  -3  -3  -3  -4  -4  -5  -4  -4  -4]
 [-10  -8  -6  -4  -2  -3  -4  -4  -5  -5  -5  -3  -4]
 [-11  -9  -7  -5  -3  -1  -2  -3  -4  -5  -4  -4  -2]]
=== SW score matrix ===
[[0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 0 0 0 0 0]
 [0 0 0 1 0 1 0 2 1 0 1 0 1]
 [0 1 0 0 0 0 0 1 3 2 1 0 0]
 [0 0 2 1 0 0 0 0 2 2 1 0 0]
 [0 1 1 1 0 0 0 0 1 3 2 1 0]
 [0 0 0 2 1 1 0 1 0 2 4 3 2]
 [0 0 1 1 1 0 0 0 0 1 3 3 2]
 [0 0 1 0 0 0 0 0 0 0 2 2 2]
 [0 0 1 0 0 0 0 0 0 0 1 1 1]
 

## Traceback (Needleman-Wunsch)

Now, let's do traceback on the NW matrix. Notice the pattern of decisions spans the entire matrix.

In [34]:
paths = traceback(a, b, score_matrix_1.shape[0] - 1, score_matrix_1.shape[1] - 1, score_matrix_1, toprint=True)
for path in paths:
    for seq in path:
        path[seq] = "".join(path[seq])
print(paths)

[[' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ']
 [' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ']
 [' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ']
 [' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ']
 [' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ']
 [' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ']
 [' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ']
 [' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ']
 [' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ']
 [' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ']
 [' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ']
 [' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' '-2']]
[[' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ']
 [' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ']
 [' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ']
 [' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ']
 [' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ']
 [' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ']
 [' ' ' 

## Traceback (Smith-Waterman)

Notice that the local alignment algorithm focuses on a subset of the overall sequence.

In [35]:
smith_waterman_sm(a, b)

('GATNTA',
 'GAT-TA',
 array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 1, 0, 1, 0, 2, 1, 0, 1, 0, 1],
        [0, 1, 0, 0, 0, 0, 0, 1, 3, 2, 1, 0, 0],
        [0, 0, 2, 1, 0, 0, 0, 0, 2, 2, 1, 0, 0],
        [0, 1, 1, 1, 0, 0, 0, 0, 1, 3, 2, 1, 0],
        [0, 0, 0, 2, 1, 1, 0, 1, 0, 2, 4, 3, 2],
        [0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 3, 3, 2],
        [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2],
        [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1],
        [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 2, 1],
        [0, 0, 0, 1, 0, 2, 1, 1, 0, 0, 1, 1, 3]]))

In [36]:
align_sequences(a, b, nw=0)

['GATNTA', 'GAT-TA']

### Example from Marcus's notebook

In [37]:
seq1 = 'TACTTAG'
seq2 = 'CACATTAA'

x = smith_waterman_sm(seq1, seq2)
y = align_sequences(seq1, seq2, nw=0)
z = smith_waterman_tth(seq1, seq2)

print(x)
print(y)
print(z)

('AC-TTA', 'ACATTA', array([[0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 1, 1, 0, 0],
       [0, 0, 1, 0, 1, 0, 0, 2, 1],
       [0, 1, 0, 2, 1, 0, 0, 1, 1],
       [0, 0, 0, 1, 1, 2, 1, 0, 0],
       [0, 0, 0, 0, 0, 2, 3, 2, 1],
       [0, 0, 1, 0, 1, 1, 2, 4, 3],
       [0, 0, 0, 0, 0, 0, 1, 3, 3]]))
['AC-TTA', 'ACATTA']
('AC-TTA', 'ACATTA', array([[0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 1, 1, 0, 0],
       [0, 0, 1, 0, 1, 0, 0, 2, 1],
       [0, 1, 0, 2, 1, 0, 0, 1, 1],
       [0, 0, 0, 1, 1, 2, 1, 0, 0],
       [0, 0, 0, 0, 0, 2, 3, 2, 1],
       [0, 0, 1, 0, 1, 1, 2, 4, 3],
       [0, 0, 0, 0, 0, 0, 1, 3, 3]]))


## Testing Speed

Here, we test speed with Claude-generated long sequences. The sequences are encoded to remove the encoding process from execution time. Note that by using a recursive structure, the fast algorithm is able to recover multiple possible alignments in a fraction of the time.

Here, we noticed a discrepancy between our algorithm output. Initially, this was caused with an initialization bug. I was using np.int8 for the scoring matrix assuming that scores wouldn't get that large. This was boosted to np.int64. Next, the homopolymer run specifically causes a combinatorial explosion for my algorithm. Since it is trying to find all possible alignments, increasing the length of sequence by just a few characters causes the algorithm to asymptotically reach infinite runtime. You can see this in the results below.

In [38]:
seq1 = (
    "ATGCGTACGTTACGATCGATCGATCGAATCGATCGATCGATCGATCG"
    "GCTAGCTAGCTAGCTAGCTAGCATGCATGCATGCATGCATGCATGCA"
    "TTACGGCATGCAATCGATCGTAGCTAGCTAGCTAGCTAGCTAGCATG"
    "AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA"  # homopolymer run
    "CAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAG"  # tandem repeat
    "GCGATCGATCGATCGTAGCTAGCTAGCTAGCTAGCTAGCTAGCTATCG"
    "ATGCATGCATGCATGCATGCATGCATGCATGCATGCATGCATGCATGC"
    "TTACGGCATGCAATCGATCGTAGCTAGCTAGCTAGCTAGCTAGCATGC"
    "CGTAGCTAGCTAGCATGCATGCATGCATGCATGCATGCATGCATGCGG"
    "ATGATGATGATGATGATGATGATGATGATGATGATGATGATGATGATGA"
)

seq2 = (
    "ATGCGTACGTTACGATCGATCGATCGAATCGATCGATCGTTCGATCG"   # SNP near end
    "GCTAGCTAGCTAGCTAGCTAGCATGCATGCATGCATTTGCATGCATGCA"  # insertion
    "TTACGGCATGCAATCGATCGTAGCAAGCTAGCAAGCTAGCTAGCATG"    # two SNPs
    "AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA"     # shorter homopolymer (deletion)
    "CAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAG"  # repeat expansion
    "GCGATCGATCGATCGTAGCTAGCTAGCTAGCTAGCTAGCTAGCTATCG"   # identical
    "ATGCATGCATGCATGCATGCATGCATGCATGCATGCATGCATGCATGC"   # identical
    "TTACGGCATGCAATCGATCGTAGCTAGCTAGCTAGCTAGCTAGCATGC"   # identical
    "CGTAGCTAGCTAGCATGCATGCATGCATGCATGCATGCATGCATGCGG"   # identical
    "ATGATGATGAAATGATGATGATGATGATGATGTTGATGATGATGATGA"    # two indels
)

seq1_encoded = encode_seq(seq1)
seq2_encoded = encode_seq(seq2)

In [39]:
t0 = time()
result_a = smith_waterman_sm(seq1, seq2)
print(f"Smith_Waterman_SM: {time() - t0}s")
pprint(result_a)

Smith_Waterman_SM: 0.3844876289367676s
('ATGCGTACGTTACGATCGATCGATCGAATCGATCGATCGATCGATCGGCTAGCTAGCTAGCTAGCTAGCATGCATGCATGCA--TGCATGCATGCATTACGGCATGCAATCGATCGTAGCTAGCTAGCTAGCTAGCTAGCATG---AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA-A--A--A--A-CAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGGCGATCGATCGATCGTAGCTAGCTAGCTAGCTAGCTAGCTAGCTATCGATGCATGCATGCATGCATGCATGCATGCATGCATGCATGCATGCATGCTTACGGCATGCAATCGATCGTAGCTAGCTAGCTAGCTAGCTAGCATGCCGTAGCTAGCTAGCATGCATGCATGCATGCATGCATGCATGCATGCGGATGATGATGATGATGATGATGATGATGATGATGATGATGATGATGATGA',
 'ATGCGTACGTTACGATCGATCGATCGAATCGATCGATCGTTCGATCGGCTAGCTAGCTAGCTAGCTAGCATGCATGCATGCATTTGCATGCATGCATTACGGCATGCAATCGATCGTAGCAAGCTAGCAAGCTAGCTAGCATGAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGGCGATCGATCGATCGTAGCTAGCTAGCTAGCTAGCTAGCTAGCTATCGATGCATGCATGCATGCATGCATGCATGCATGCATGCATGCATGCATGCTTACGGCATGCAATCGATCGTAGCTAGCTAGCTAGCTAGCTAGCATGCCGTAGCTAGCTAGCATGCATGCATGCATGCATGCATGCATGCATGCGGATGATGATGA-AATGATGATG

In [40]:
t0 = time()
result_b = smith_waterman_tth(seq1, seq2)
print(f"Smith_Waterman_TTH: {time() - t0}s")
pprint(result_b)

Smith_Waterman_TTH: 0.20491313934326172s
('ATGCGTACGTTACGATCGATCGATCGAATCGATCGATCGATCGATCGGCTAGCTAGCTAGCTAGCTAGCATGCATGCATGCAT--GCATGCATGCATTACGGCATGCAATCGATCGTAGCTAGCTAGCTAGCTAGCTAGCATGAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA-A-CAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAG--GC-G-A-----TCGATCGATCGTAGCTAGCTAGCTAGCTAGCTAGCTAGCTATCGATGCATGCATGCATGCATGCATGCATGCATGCATGCATGCATGCATGCTTACGGCATGCAATCGATCGTAGCTAGCTAGCTAGCTAGCTAGCATGCCGTAGCTAGCTAGCATGCATGCATGCATGCATGCATGCATGCATGCGGATGATGATGATGATGATGATGATGATGATGATGATGATGATGATGATGA',
 'ATGCGTACGTTACGATCGATCGATCGAATCGATCGATCGTTCGATCGGCTAGCTAGCTAGCTAGCTAGCATGCATGCATGCATTTGCATGCATGCATTACGGCATGCAATCGATCGTAGCAAGCTAGCAAGCTAGCTAGCATGAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGGCGATCGATCGATCGTAGCTAGCTAGCTAGCTAGCTAGCTAGCTATCGATGCATGCATGCATGCATGCATGCATGCATGCATGCATGCATGCATGCTTACGGCATGCAATCGATCGTAGCTAGCTAGCTAGCTAGCTAGCATGCCGTAGCTAGCTAGCATGCATGCATGCATGCATGCATGCATGCATGCGGATGATGATGAA-ATGATGA

In [41]:
t0 = time()
print("\n191-length sequence")
result_c = align_sequences(seq1_encoded[:191], seq2_encoded[:191], nw=0)
print(f"Smith_Waterman_Fast, {time() - t0}s")
print(f"{len(result_c)} possible alignments!\n")

t0 = time()
print("\n192-length sequence")
result_c = align_sequences(seq1_encoded[:192], seq2_encoded[:192], nw=0)
print(f"Smith_Waterman_Fast, {time() - t0}s")
print(f"{len(result_c)} possible alignments!\n")
pprint(result_c[:10])


191-length sequence
Smith_Waterman_Fast, 0.005522489547729492s
6 possible alignments!


192-length sequence
Smith_Waterman_Fast, 0.026454687118530273s
288 possible alignments!

['ATGCGTACGTTACGATCGATCGATCGAATCGATCGATCGATCGATCGGCTAGCTAGCTAGCTAGCTAGCATGCATGCATGCAT--GCATGCATGCATTACGGCATGCAATCGATCGTAGCTAGCTAGCTAGCTAGCTAGCATGAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACA',
 'ATGCGTACGTTACGATCGATCGATCGAATCGATCGATCGTTCGATCGGCTAGCTAGCTAGCTAGCTAGCATGCATGCATGCATTTGCATGCATGCATTACGGCATGCAATCGATCGTAGCAAGCTAGCAAGCTAGCTAGCATGAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA-CA',
 'ATGCGTACGTTACGATCGATCGATCGAATCGATCGATCGATCGATCGGCTAGCTAGCTAGCTAGCTAGCATGCATGCATGCA-T-GCATGCATGCATTACGGCATGCAATCGATCGTAGCTAGCTAGCTAGCTAGCTAGCATGAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACA',
 'ATGCGTACGTTACGATCGATCGATCGAATCGATCGATCGTTCGATCGGCTAGCTAGCTAGCTAGCTAGCATGCATGCATGCATTTGCATGCATGCATTACGGCATGCAATCGATCGTAGCAAGCTAGCAAGCTAGCTAGCATGAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA-CA',
 'ATGCGTACGTTACGATCGATCGATCGAA

## Now, trying Needleman-Wunsch

The time here is slightly longer, but the algorithm actually fails with a sequence of length 180. This is likely because the homopolymer region is being sliced, yielding a combinatorial explosion in the possible alignments. In the output below, you can actually see all the possible alignments the algorithm is scoring. For every additional alignment beyond 192, these are compounded. We're entering exponential time.

In [42]:
t0 = time()
print("\n191-length sequence")
result_c = align_sequences(seq1_encoded[:191], seq2_encoded[:191], nw=1)
print(f"Needleman_Wunsch_Fast, {time() - t0}s")
print(f"{len(result_c)} possible alignments!\n")

t0 = time()
print("\n192-length sequence")
result_c = align_sequences(seq1_encoded[:192], seq2_encoded[:192], nw=1)
print(f"Needleman_Wunsch_Fast, {time() - t0}s")
print(f"{len(result_c)} possible alignments!\n")
pprint(result_c[:10])


191-length sequence
Needleman_Wunsch_Fast, 0.028435468673706055s
288 possible alignments!


192-length sequence
Needleman_Wunsch_Fast, 0.02528667449951172s
288 possible alignments!

['ATGCGTACGTTACGATCGATCGATCGAATCGATCGATCGATCGATCGGCTAGCTAGCTAGCTAGCTAGCATGCATGCATGCAT--GCATGCATGCATTACGGCATGCAATCGATCGTAGCTAGCTAGCTAGCTAGCTAGCATGAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACAG',
 'ATGCGTACGTTACGATCGATCGATCGAATCGATCGATCGTTCGATCGGCTAGCTAGCTAGCTAGCTAGCATGCATGCATGCATTTGCATGCATGCATTACGGCATGCAATCGATCGTAGCAAGCTAGCAAGCTAGCTAGCATGAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA-CA-',
 'ATGCGTACGTTACGATCGATCGATCGAATCGATCGATCGATCGATCGGCTAGCTAGCTAGCTAGCTAGCATGCATGCATGCA-T-GCATGCATGCATTACGGCATGCAATCGATCGTAGCTAGCTAGCTAGCTAGCTAGCATGAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACAG',
 'ATGCGTACGTTACGATCGATCGATCGAATCGATCGATCGTTCGATCGGCTAGCTAGCTAGCTAGCTAGCATGCATGCATGCATTTGCATGCATGCATTACGGCATGCAATCGATCGTAGCAAGCTAGCAAGCTAGCTAGCATGAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA-CA-',
 'ATGCGTACGTTACGATCGA

## Different Sequence

It was at this point when we realized that the fast function is primarily limited by combinatorial explosion. For this particular sequence > length 150, the number of possible branches for certain sequences becomes so large that the algorithm never finishes.

You can see that there is a long list of sequences here.

Notice the 

In [43]:
import random
random.seed(42)

bases = "ATGC"
background = "".join(random.choice(bases) for _ in range(500))

seq1 = background

seq2 = (
    background[:80]          +   # identical
    background[81:160]       +   # deletion at pos 80
    "TTTTGGCC"               +   # 8 base insertion
    background[160:260]      +   # identical
    "GCGCATAT"               +   # 4 SNPs
    background[268:360]      +   # identical (skipping 260-268, deletion)
    background[361:440]      +   # deletion at pos 360
    "AAGTCCAG"               +   # 4 base insertion
    background[440:]             # identical to end
)

In [44]:
t0 = time()
print("\n500-length sequence")
result_a = smith_waterman_sm(seq1, seq2)
print(f"Smith_Waterman_SM: {time() - t0}s")
pprint(result_a)

t0 = time()
print("\n500-length sequence")
result_b = smith_waterman_tth(seq1, seq2)
print(f"Smith_Waterman_TTH: {time() - t0}s")
pprint(result_b)


500-length sequence
Smith_Waterman_SM: 0.30689501762390137s
('AAGTTTAACAAATTATCTCGATCGGTTGAACAGGGACACAGGTAATGATACGCGTGGTGATTTCCGTGATAGCGATGTCCCTGTTGCCGTTCAAATTCACCCGAAGGAGCTCAGTAGTTGTAGCAAGGTATAACATTCTGCTTGCGCCATTAGATTAAA----T----AAGATGCTTCTCCTAACGCCCAAACGATTTCTCTGCTACAAAATTCCCTCATCACGCGCCTTGTAAGAACTAATAATCATAACGGTGTGCTGCGAACAATGTG-ATGGTCGAGATGAATGGTGTGCGAACGAAGTGTCCAAATAGTCTAGGAGTTAGCTTTTCATGCTGTACACTTCGGTTATCGGAGGCGAAGTGAACGGCACTGACATGCAGGAGGCGCGTTCCTGCAGGTCGCCCTCTAGGATGTTTAATCACCTCCCTTAACTTCATACTCG-CC-C---T-C--CGTGCTGCAGTGGGATTTCTTACCGCCATCCACCAGGCCTCTGCCACGCTCTACAACTCT',
 'AAGTTTAACAAATTATCTCGATCGGTTGAACAGGGACACAGGTAATGATACGCGTGGTGATTTCCGTGATAGCGATGT-CCTGTTGCCGTTCAAATTCACCCGAAGGAGCTCAGTAGTTGTAGCAAGGTATAACATTCTGCTTGCGCCATTAGATTAAATTTTTGGCCAAGATGCTTCTCCTAACGCCCAAACGATTTCTCTGCTACAAAATTCCCTCATCACGCGCCTTGTAAGAACTAATAATCATAACGGTGTGCTGCGAACAATGCGCAT-ATCGAGATGAATGGTGTGCGAACGAAGTGTCCAAATAGTCTAGGAGTTAGCTTTTCATGCTGTACACTTCGGTTATCGGAGGCGAAGTGAACGG-ACTGACATGCAGGAGGCGCGTTCCTGCAGGTCGCCCTCTAGGATG

In [45]:

t0 = time()
print("\n500-length sequence")
result_c = align_sequences(seq1, seq2, nw=0)
print(f"\Smith_Waterman_Fast, {time() - t0}s")
print(f"{len(result_c)} possible alignments!\n")

pprint(result_c[:10])


500-length sequence
\Smith_Waterman_Fast, 0.3175389766693115s
1500 possible alignments!

['AAGTTTAACAAATTATCTCGATCGGTTGAACAGGGACACAGGTAATGATACGCGTGGTGATTTCCGTGATAGCGATGTCCCTGTTGCCGTTCAAATTCACCCGAAGGAGCTCAGTAGTTGTAGCAAGGTATAACATTCTGCTTGCGCCATTAGATTAAAT--------AAGATGCTTCTCCTAACGCCCAAACGATTTCTCTGCTACAAAATTCCCTCATCACGCGCCTTGTAAGAACTAATAATCATAACGGTGTGCTGCGAACAATGTG-ATGGTCGAGATGAATGGTGTGCGAACGAAGTGTCCAAATAGTCTAGGAGTTAGCTTTTCATGCTGTACACTTCGGTTATCGGAGGCGAAGTGAACGGCACTGACATGCAGGAGGCGCGTTCCTGCAGGTCGCCCTCTAGGATGTTTAATCACCTCCCTTAACTTCATACTCGCCCTC----C--G--TGCTGCAGTGGGATTTCTTACCGCCATCCACCAGGCCTCTGCCACGCTCTACAACTCT',
 'AAGTTTAACAAATTATCTCGATCGGTTGAACAGGGACACAGGTAATGATACGCGTGGTGATTTCCGTGATAGCGATGTCC-TGTTGCCGTTCAAATTCACCCGAAGGAGCTCAGTAGTTGTAGCAAGGTATAACATTCTGCTTGCGCCATTAGATTAAATTTTTGGCCAAGATGCTTCTCCTAACGCCCAAACGATTTCTCTGCTACAAAATTCCCTCATCACGCGCCTTGTAAGAACTAATAATCATAACGGTGTGCTGCGAACAATGCGCATA-TCGAGATGAATGGTGTGCGAACGAAGTGTCCAAATAGTCTAGGAGTTAGCTTTTCATGCTGTACACTTCGGTTATCGGAGGCGAAGTGAACGG-ACTGACATGCAGGAGG

In [46]:

t0 = time()
print("\n400-length sequence")
result_c = align_sequences(seq1[:400], seq2[:400], nw=1)
print(f"\nNeedleman_Wunsch_Fast, {time() - t0}s")
print(f"{len(result_c)} possible alignments!\n")

t0 = time()
print("\n500-length sequence")
result_c = align_sequences(seq1, seq2, nw=1)
print(f"\nNeedleman_Wunsch_Fast, {time() - t0}s")
print(f"{len(result_c)} possible alignments!\n")

pprint(result_c[:10])


400-length sequence

Needleman_Wunsch_Fast, 0.025516510009765625s
60 possible alignments!


500-length sequence

Needleman_Wunsch_Fast, 0.3378298282623291s
1500 possible alignments!

['AAGTTTAACAAATTATCTCGATCGGTTGAACAGGGACACAGGTAATGATACGCGTGGTGATTTCCGTGATAGCGATGTCCCTGTTGCCGTTCAAATTCACCCGAAGGAGCTCAGTAGTTGTAGCAAGGTATAACATTCTGCTTGCGCCATTAGATTAAAT--------AAGATGCTTCTCCTAACGCCCAAACGATTTCTCTGCTACAAAATTCCCTCATCACGCGCCTTGTAAGAACTAATAATCATAACGGTGTGCTGCGAACAATGTG-ATGGTCGAGATGAATGGTGTGCGAACGAAGTGTCCAAATAGTCTAGGAGTTAGCTTTTCATGCTGTACACTTCGGTTATCGGAGGCGAAGTGAACGGCACTGACATGCAGGAGGCGCGTTCCTGCAGGTCGCCCTCTAGGATGTTTAATCACCTCCCTTAACTTCATACTCGCCCTC----C--G--TGCTGCAGTGGGATTTCTTACCGCCATCCACCAGGCCTCTGCCACGCTCTACAACTCT',
 'AAGTTTAACAAATTATCTCGATCGGTTGAACAGGGACACAGGTAATGATACGCGTGGTGATTTCCGTGATAGCGATGTCC-TGTTGCCGTTCAAATTCACCCGAAGGAGCTCAGTAGTTGTAGCAAGGTATAACATTCTGCTTGCGCCATTAGATTAAATTTTTGGCCAAGATGCTTCTCCTAACGCCCAAACGATTTCTCTGCTACAAAATTCCCTCATCACGCGCCTTGTAAGAACTAATAATCATAACGGTGTGCTGCGAACAATGCGCATA-TCGAGATGAATGGTGT

## Conclusion

Our tests revealed that the structure of different sequences has a tremendous impact on the number of possible alignments, and this cannot be overcome with greater efficiency. Specifically, regions with many repeats are liable to cause this explosion. To avoid this, a repeat-avoidance mechanism could be implmeneted to prevent the scoring matrix from reflecting these situations. The exponential growth in possible alignments means that a heuristic is necessary.